In [ ]:
# Install once if needed: %pip install torch torchvision matplotlib pandas tqdm
import copy, math, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__)

In [ ]:
def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    dataset: str = 'CIFAR10'
    n_tasks: int = 5
    buffer_size: int = 200
    epochs: int = 50
    batch_size: int = 128
    minibatch_size: int = 128
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    empty_probability: float = 0.9
    alpha: float = 1.0       # L_ide
    beta: float = 1.0        # L_rep-ice
    class_balance: bool = False
    num_workers: int = 2
    data_root: str = './data'
    gcil_mode: str = 'uniform'

RUN_CONFIG = Config()
seed_everything(0)
print(asdict(RUN_CONFIG))

## 1. Class-incremental data stream

CIFAR-10 uses 5 tasks with 2 classes per task; CIFAR-100 uses 10 tasks with 10 classes per task. The task identity is not given to the model at test time.

In [ ]:
def make_transforms(train=True):
    ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()] if train else []
    ops += [transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616))]
    return transforms.Compose(ops)

class TinyImageNetVal(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        from PIL import Image
        self.root=Path(root); self.transform=transform; self.records=[]
        wnids=(self.root/'wnids.txt').read_text().splitlines(); self.class_to_idx={c:i for i,c in enumerate(wnids)}
        ann=self.root/'val'/'val_annotations.txt'
        for line in ann.read_text().splitlines():
            fn, cls, *_ = line.split(); self.records.append((self.root/'val'/'images'/fn, self.class_to_idx[cls]))
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        from PIL import Image
        p,y=self.records[i]; x=Image.open(p).convert('RGB'); return (self.transform(x) if self.transform else x), y

class BenchmarkStream:
    """Paper datasets: Split CIFAR-10/100, Split TinyImageNet, and GCIL-CIFAR-100."""
    def __init__(self, cfg):
        name=cfg.dataset.upper(); self.cfg=cfg
        if name in ('CIFAR10','CIFAR100'):
            Dataset, self.classes = (datasets.CIFAR10,10) if name=='CIFAR10' else (datasets.CIFAR100,100)
            self.train=Dataset(cfg.data_root, train=True, download=True, transform=make_transforms(True)); self.test=Dataset(cfg.data_root, train=False, download=True, transform=make_transforms(False))
            self.train_targets=np.asarray(self.train.targets); self.test_targets=np.asarray(self.test.targets)
        elif name in ('TINYIMAGENET','TINY-IMAGENET'):
            root=Path(cfg.data_root)/'tiny-imagenet-200'; self.classes=200
            self.train=datasets.ImageFolder(root/'train', transform=transforms.Compose([transforms.RandomCrop(64,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((.4802,.4481,.3975),(.2302,.2265,.2262))]))
            self.test=TinyImageNetVal(root, transforms.Compose([transforms.ToTensor(),transforms.Normalize((.4802,.4481,.3975),(.2302,.2265,.2262))]))
            self.train_targets=np.asarray(self.train.targets); self.test_targets=np.asarray([y for _,y in self.test.records])
        else: raise ValueError('dataset must be CIFAR10, CIFAR100, or TinyImageNet')
        self.n_tasks=cfg.n_tasks; self.classes_per_task=self.classes//cfg.n_tasks
        if name.startswith('CIFAR100') and cfg.gcil_mode in ('uniform','longtail'):
            self.task_classes=self._make_gcil_classes()
        else: self.task_classes=[list(range(t*self.classes_per_task,(t+1)*self.classes_per_task)) for t in range(cfg.n_tasks)]

    def _make_gcil_classes(self):
        rng=np.random.default_rng(0); out=[]
        for t in range(self.n_tasks):
            # GCIL permits overlap; each task still observes a variable class subset.
            k=int(rng.integers(max(2,self.classes_per_task//2), self.classes_per_task+1)); out.append(sorted(rng.choice(self.classes,k,replace=False).tolist()))
        return out

    def loaders(self, task, batch_size, workers=2):
        cls=self.task_classes[task]; tr_idx=np.flatnonzero(np.isin(self.train_targets,cls)); te_idx=np.flatnonzero(np.isin(self.test_targets,cls))
        if self.cfg.dataset.upper()=='CIFAR100' and self.cfg.gcil_mode=='longtail':
            y=self.train_targets[tr_idx]; keep=[]; counts={c:max(10,int(5000/(2**i))) for i,c in enumerate(sorted(cls))}
            for c in cls: keep.extend(tr_idx[y==c][:counts[c]])
            tr_idx=np.asarray(keep)
        tr=DataLoader(Subset(self.train,tr_idx),batch_size=batch_size,shuffle=True,num_workers=workers,pin_memory=True)
        te=DataLoader(Subset(self.test,te_idx),batch_size=batch_size,shuffle=False,num_workers=workers,pin_memory=True)
        return tr,te

# Real paper data stream examples:
# stream = BenchmarkStream(Config(dataset='CIFAR10', n_tasks=5, epochs=50, buffer_size=200))
# stream = BenchmarkStream(Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500))
# stream = BenchmarkStream(Config(dataset='TinyImageNet', n_tasks=10, epochs=100, buffer_size=4000))
# stream = BenchmarkStream(Config(dataset='CIFAR100', n_tasks=10, gcil_mode='longtail', buffer_size=500))

In [ ]:
class ReplayBuffer:
    """Reservoir replay with an optional class-balancing replacement rule."""
    def __init__(self, capacity, device, class_balance=False):
        self.capacity, self.device, self.class_balance = capacity, device, class_balance
        self.examples, self.labels = [], []
        self.seen = 0

    def __len__(self): return len(self.labels)
    def is_empty(self): return len(self) == 0

    def _index(self, label):
        if self.seen < self.capacity: return self.seen
        if self.class_balance and self.labels:
            counts = np.bincount(np.asarray(self.labels), minlength=int(max(self.labels))+1)
            candidates = np.flatnonzero(np.isin(self.labels, np.flatnonzero(counts == counts.max())))
            if np.random.randint(self.seen + 1) < self.capacity: return int(np.random.choice(candidates))
            return -1
        j = np.random.randint(self.seen + 1)
        return int(j) if j < self.capacity else -1

    def add(self, examples, labels):
        for x, y in zip(examples.detach().cpu(), labels.detach().cpu()):
            idx = self._index(int(y)); self.seen += 1
            if idx < 0: continue
            if idx == len(self.examples): self.examples.append(x.clone()); self.labels.append(int(y))
            else: self.examples[idx] = x.clone(); self.labels[idx] = int(y)

    def sample(self, n, transform):
        n = min(n, len(self)); ids = np.random.choice(len(self), n, replace=False)
        x = torch.stack([transform(self.examples[i]) for i in ids]).to(self.device)
        y = torch.tensor([self.labels[i] for i in ids], device=self.device)
        return x, y

In [ ]:
class IdempotentResNet(nn.Module):
    def __init__(self, num_classes, nf=32):
        super().__init__(); self.num_classes = num_classes
        def block(cin, cout, stride):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                                 nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU())
        self.f1 = nn.Sequential(nn.Conv2d(3,nf,3,1,1,bias=False), nn.BatchNorm2d(nf), nn.ReLU(),
                               block(nf,nf,1), block(nf,nf*2,2), block(nf*2,nf*4,2), block(nf*4,nf*8,2),
                               nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.feature_dim = nf*8
        self.label_feature = nn.Sequential(nn.Linear(num_classes, self.feature_dim), nn.LeakyReLU(0.1))
        self.f2 = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x, second_input):
        z = self.f1(x) + self.label_feature(second_input)
        return self.f2(z)

    def first_pass(self, x):
        empty = torch.full((x.size(0), self.num_classes), 1/self.num_classes, device=x.device)
        return self(x, empty)

def one_hot_or_empty(labels, num_classes, p_empty, device):
    use_empty = torch.rand((), device=device) < p_empty
    if use_empty: return torch.full((labels.size(0), num_classes), 1/num_classes, device=device)
    return F.one_hot(labels, num_classes=num_classes).float()

In [ ]:
def ider_step(model, old_model, optimizer, x, y, buffer, cfg, transform):
    model.train(); optimizer.zero_grad(); C = model.num_classes
    y_star = one_hot_or_empty(y, C, cfg.empty_probability, x.device)
    y0 = model(x, y_star); y1 = model(x, y0.softmax(-1))
    loss_ice = 0.5 * (F.cross_entropy(y0, y) + F.cross_entropy(y1, y))
    loss_ide = torch.zeros((), device=x.device); loss_rep = torch.zeros((), device=x.device)

    if old_model is not None and not buffer.is_empty() and cfg.alpha:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        empty = torch.full((bx.size(0), C), 1/C, device=x.device)
        current_pred = model(bx, empty)
        with torch.no_grad(): stable_pred = old_model(bx, current_pred.softmax(-1))
        loss_ide = F.mse_loss(current_pred, stable_pred)

    if not buffer.is_empty() and cfg.beta:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        by_star = one_hot_or_empty(by, C, cfg.empty_probability, x.device)
        br0 = model(bx, by_star); br1 = model(bx, br0.softmax(-1))
        loss_rep = F.cross_entropy(br0, by) + F.cross_entropy(br1, by)

    total = loss_ice + cfg.alpha*loss_ide + cfg.beta*loss_rep
    total.backward(); optimizer.step(); buffer.add(x, y)
    return {
        'total': float(total.detach()), 'L_ice': float(loss_ice.detach()),
        'L_ide': float(loss_ide.detach()), 'L_rep-ice': float(loss_rep.detach())
    }

In [ ]:
@torch.no_grad()
def evaluate(model, loaders, device):
    model.eval(); scores = []
    for loader in loaders:
        correct = total = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model.first_pass(x).argmax(1)
            correct += int((pred == y).sum()); total += y.numel()
        scores.append(100*correct/max(total,1))
    return scores

def expected_calibration_error(model, loader, device, bins=15):
    model.eval(); confs=[]; correct=[]
    with torch.no_grad():
        for x,y in loader:
            p = model.first_pass(x.to(device)).softmax(1); c, pred = p.max(1)
            confs.append(c.cpu()); correct.append(pred.cpu().eq(y))
    conf, cor = torch.cat(confs), torch.cat(correct).float(); ece = torch.zeros(())
    for lo, hi in zip(torch.linspace(0,1,bins+1)[:-1], torch.linspace(0,1,bins+1)[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any(): ece += mask.float().mean() * (cor[mask].mean() - conf[mask].mean()).abs()
    return float(ece*100)

def final_metrics(history):
    final = np.asarray(history[-1], dtype=float); faa = final.mean()
    # Histories are ragged: task j only exists from the moment it is learned.
    peaks = []
    for j, final_score in enumerate(final):
        observed = [row[j] for row in history if len(row) > j]
        peaks.append(max(observed))
    forgetting = np.mean(np.asarray(peaks) - final)
    return {'FAA': float(faa), 'FF': float(forgetting)}

In [ ]:
def run_ider(cfg, stream=None, device=DEVICE):
    seed_everything(0)
    if stream is None: stream = BenchmarkStream(cfg)
    model = IdempotentResNet(stream.classes).to(device)
    old_model = None; buffer = ReplayBuffer(cfg.buffer_size, device, cfg.class_balance)
    history, loss_rows, test_loaders = [], [], []
    for task in range(cfg.n_tasks):
        train_loader, test_loader = stream.loaders(task, cfg.batch_size, cfg.num_workers)
        test_loaders.append(test_loader)
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1,cfg.epochs*len(train_loader)))
        for epoch in range(cfg.epochs):
            for x,y in tqdm(train_loader, desc=f'task {task+1}/{cfg.n_tasks}, epoch {epoch+1}', leave=False):
                x,y = x.to(device), y.to(device)
                row = ider_step(model, old_model, optimizer, x, y, buffer, cfg, stream.train.transform)
                row.update(task=task, epoch=epoch); loss_rows.append(row); scheduler.step()
        scores = evaluate(model, test_loaders, device); history.append(scores)
        print(f'task {task+1}: FAA over seen tasks = {np.mean(scores):.2f}% | scores = {[round(s,2) for s in scores]}')
        old_model = copy.deepcopy(model).eval()
        for p in old_model.parameters(): p.requires_grad_(False)
    metrics = final_metrics(history)
    metrics['ECE_last_task'] = expected_calibration_error(model, test_loaders[-1], device)
    return model, history, pd.DataFrame(loss_rows), metrics

## 5. Paper training runs

The main path uses the real benchmark datasets below. No synthetic data is used for reported results.

In [ ]:
# Split CIFAR-10: 5 tasks, 50 epochs/task, buffer 200 or 500.
cifar10_cfg = Config(dataset='CIFAR10', n_tasks=5, epochs=50, buffer_size=200)
# cifar10_stream = BenchmarkStream(cifar10_cfg)
# cifar10_model, cifar10_history, cifar10_losses, cifar10_metrics = run_ider(cifar10_cfg, cifar10_stream, DEVICE)

# Split CIFAR-100: 10 tasks, 50 epochs/task, buffer 500 or 2000.
cifar100_cfg = Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500)
# cifar100_stream = BenchmarkStream(cifar100_cfg)
# cifar100_model, cifar100_history, cifar100_losses, cifar100_metrics = run_ider(cifar100_cfg, cifar100_stream, DEVICE)

# Split TinyImageNet: 10 tasks, 100 epochs/task, buffer 4000.
tiny_cfg = Config(dataset='TinyImageNet', n_tasks=10, epochs=100, buffer_size=4000, batch_size=128)
# tiny_stream = BenchmarkStream(tiny_cfg)
# tiny_model, tiny_history, tiny_losses, tiny_metrics = run_ider(tiny_cfg, tiny_stream, DEVICE)

# GCIL-CIFAR-100: overlapping classes; switch gcil_mode to 'longtail' for imbalanced tasks.
gcil_cfg = Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500, gcil_mode='uniform')
# gcil_stream = BenchmarkStream(gcil_cfg)
# gcil_model, gcil_history, gcil_losses, gcil_metrics = run_ider(gcil_cfg, gcil_stream, DEVICE)

In [ ]:
# Recommended starting run: one seed, one epoch, CIFAR-10.
full_cfg = Config(dataset='CIFAR10', n_tasks=5, buffer_size=200, epochs=50, class_balance=False)
# stream = BenchmarkStream(full_cfg)
# model, history, losses, metrics = run_ider(full_cfg, stream, DEVICE)
# print(metrics)
# pd.DataFrame(history, columns=[f'task_{i+1}' for i in range(len(history[-1]))]).plot(marker='o', ylim=(0,100), figsize=(8,4))
# plt.ylabel('Accuracy (%)'); plt.xlabel('Training task'); plt.title('IDER task accuracy'); plt.show()

## 6. BFP+ID and CLS-ER+ID (paper plug-ins)

Appendix D.3 defines `L_BFP = ||A h_t(x,0) - h_{t-1}(x,0)||²` and `L_BFP+ID = L_ice + αL_ide + βL_rep-ice + γL_BFP`. CLS-ER+ID keeps fast and slow EMA semantic memories and adds their confidence-selected consistency target.

In [ ]:
class EMA:
    def __init__(self, model, decay): self.model=copy.deepcopy(model).eval(); self.decay=decay
    @torch.no_grad()
    def update(self, source):
        for q,p in zip(self.model.parameters(), source.parameters()): q.mul_(self.decay).add_(p, alpha=1-self.decay)
        for q,p in zip(self.model.buffers(), source.buffers()): q.copy_(p)

def bfp_id_step(model, old_model, A, optimizer, x, y, buffer, cfg, transform, gamma=1.0):
    model.train(); A.train(); optimizer.zero_grad(); C=model.num_classes
    ys=one_hot_or_empty(y,C,cfg.empty_probability,x.device); y0=model(x,ys); y1=model(x,y0.softmax(-1))
    l_ice=.5*(F.cross_entropy(y0,y)+F.cross_entropy(y1,y)); l_ide=torch.zeros((),device=x.device); l_rep=torch.zeros((),device=x.device); l_bfp=torch.zeros((),device=x.device)
    if old_model is not None and not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); empty=torch.full((len(bx),C),1/C,device=x.device); cur=model(bx,empty)
        with torch.no_grad(): stable=old_model(bx,cur.softmax(-1))
        l_ide=F.mse_loss(cur,stable); l_bfp=F.mse_loss(A(model.f1(bx)),old_model.f1(bx).detach())
    if not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); bs=one_hot_or_empty(by,C,cfg.empty_probability,x.device); r0=model(bx,bs); r1=model(bx,r0.softmax(-1)); l_rep=F.cross_entropy(r0,by)+F.cross_entropy(r1,by)
    total=l_ice+cfg.alpha*l_ide+cfg.beta*l_rep+gamma*l_bfp; total.backward(); optimizer.step(); buffer.add(x,y)
    return {'total':float(total.detach()),'L_ice':float(l_ice.detach()),'L_ide':float(l_ide.detach()),'L_rep-ice':float(l_rep.detach()),'L_BFP':float(l_bfp.detach())}

def clser_id_step(model, old_model, plastic, stable, optimizer, x, y, buffer, cfg, transform, lam=1.0):
    row=ider_step(model,old_model,optimizer,x,y,buffer,cfg,transform)
    if not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); empty=torch.full((len(bx),model.num_classes),1/model.num_classes,device=x.device)
        with torch.no_grad(): pp=plastic.model(bx,empty); sp=stable.model(bx,empty); target=torch.where(pp.max(1,keepdim=True).values>=sp.max(1,keepdim=True).values,pp,sp)
        optimizer.zero_grad(); lc=F.mse_loss(model(bx,empty),target); (lam*lc).backward(); optimizer.step(); row['L_CLS-ER']=float(lc.detach())
    plastic.update(model); stable.update(model); return row

def run_variant(cfg, variant='ER+ID', stream=None, device=DEVICE, gamma=1.0):
    stream=BenchmarkStream(cfg) if stream is None else stream; model=IdempotentResNet(stream.classes).to(device); old=None; buffer=ReplayBuffer(cfg.buffer_size,device,cfg.class_balance); A=nn.Linear(model.feature_dim,model.feature_dim,bias=False).to(device) if variant=='BFP+ID' else None; plastic=stable=None; history=[]
    for task in range(cfg.n_tasks):
        tr,te=stream.loaders(task,cfg.batch_size,cfg.num_workers); opt_params=list(model.parameters()) + ([] if A is None else list(A.parameters())); optimizer=torch.optim.SGD(opt_params,lr=cfg.lr,momentum=cfg.momentum,weight_decay=cfg.weight_decay); scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=[35,45] if cfg.epochs==50 else [35,60,75],gamma=.1)
        if variant=='CLS-ER+ID' and plastic is None: plastic=EMA(model,.999); stable=EMA(model,.9999)
        for epoch in range(cfg.epochs):
            for x,y in tqdm(tr,desc=f'{variant} task {task+1}/{cfg.n_tasks}',leave=False):
                x,y=x.to(device),y.to(device)
                if variant=='BFP+ID': row=bfp_id_step(model,old,A,optimizer,x,y,buffer,cfg,stream.train.transform,gamma)
                elif variant=='CLS-ER+ID': row=clser_id_step(model,old,plastic,stable,optimizer,x,y,buffer,cfg,stream.train.transform)
                else: row=ider_step(model,old,optimizer,x,y,buffer,cfg,stream.train.transform)
                scheduler.step()
        test_loaders=[stream.loaders(i,cfg.batch_size,cfg.num_workers)[1] for i in range(task+1)]; scores=evaluate(model,test_loaders,device); history.append(scores); print(variant,'task',task+1,'FAA',round(float(np.mean(scores)),2)); old=copy.deepcopy(model).eval()
        for p in old.parameters(): p.requires_grad_(False)
    return model,history,final_metrics(history)

In [ ]:
def run_ablation(base_cfg, stream=None, device=DEVICE):
    rows=[]
    for name in ['ER+ID','BFP+ID','CLS-ER+ID']:
        _,_,metrics=run_variant(copy.deepcopy(base_cfg),variant=name,stream=stream,device=device)
        rows.append({'method':name,**metrics})
    return pd.DataFrame(rows)

# Real-data benchmark examples; uncomment only after setting the matching dataset path.
# result_bfp = run_variant(cifar100_cfg,'BFP+ID',cifar100_stream,DEVICE,gamma=1.0)
# result_clser = run_variant(cifar100_cfg,'CLS-ER+ID',cifar100_stream,DEVICE)
# display(run_ablation(cifar100_cfg,cifar100_stream,DEVICE))

## Reproducibility notes

Use 5 seeds for the paper tables. Split CIFAR-10/CIFAR-100 use 50 epochs per task; Split TinyImageNet uses 100. The paper uses P=0.9, SGD, and the dataset/buffer-specific learning rates from Appendix D.2. Report FAA, FF, and ECE only after all seeds finish. The official IDER repository contains ER and IDER code; BFP+ID is reproduced from Appendix D.3 and CLS-ER+ID is implemented as the CLS-ER dual-EMA plug-in described in the cited CLS-ER method.